# Étape 4 — Exploratory Data Analysis

Cette analyse explore le dataset nettoyé afin de décrire la target et les facteurs associés au churn avant toute modélisation. Les résultats décrivent des associations dans ce sample IBM fictif ; ils ne démontrent aucune relation causale.

## 1. Chargement et configuration reproductible

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from customer_churn_prediction.eda import (
    CATEGORICAL_COLUMNS, NUMERIC_COLUMNS, categorical_summary,
    generate_figures, load_data, multivariate_tables, numeric_summary,
)

DATA_PATH = PROJECT_ROOT / 'data/processed/telco_customer_churn_clean.csv'
FIGURES_DIR = PROJECT_ROOT / 'reports/figures'
data = load_data(DATA_PATH)
data.head()

## 2. Vérifications rapides
Le contrôle porte sur les dimensions, les types, les valeurs manquantes, les doublons et l'unicité de l'identifiant. Il ne modifie pas les données.

In [ ]:
print(f'Dimensions : {data.shape}')
print(f'Valeurs manquantes : {data.isna().sum().sum()}')
print(f'Doublons complets : {data.duplicated().sum()}')
print(f'customerID dupliqués : {data.customerID.duplicated().sum()}')
data.dtypes.to_frame('dtype')

## 3. Analyse de la target
Les effectifs et proportions permettent d'évaluer le déséquilibre brut, sans décider d'une méthode de rééquilibrage.

In [ ]:
target_summary = pd.concat([
    data['Churn'].value_counts().rename('clients'),
    data['Churn'].value_counts(normalize=True).rename('proportion'),
], axis=1)
display(target_summary)
generate_figures(data, FIGURES_DIR)
display(plt.imread(FIGURES_DIR / 'churn_target_distribution.png'))
plt.axis('off');

**Lecture.** Le churn concerne 1 869 clients sur 7 043 (26,54 %). La classe positive est minoritaire ; l'accuracy seule pourrait donc masquer une mauvaise détection des churners. Aucune stratégie de resampling n'est décidée ici.

## 4. Variables numériques
Nous comparons statistiques, distributions et boxplots de `tenure`, `MonthlyCharges` et `TotalCharges` selon la target.

In [ ]:
display(data[NUMERIC_COLUMNS].describe().T)
display(numeric_summary(data).round(2))
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].imshow(plt.imread(FIGURES_DIR / 'numeric_distributions_by_churn.png')); axes[0].axis('off')
axes[1].imshow(plt.imread(FIGURES_DIR / 'numeric_boxplots_by_churn.png')); axes[1].axis('off')
plt.tight_layout();

**Résultats quantitatifs.** Les churners ont une ancienneté moyenne de 17,98 mois (médiane 10), contre 37,57 mois (médiane 38) pour les non-churners. Leurs charges mensuelles moyennes sont plus élevées : 74,44 contre 61,27. Leur total cumulé moyen est toutefois plus faible : 1 531,80 contre 2 549,91, ce qui est cohérent avec leur ancienneté plus courte. Les distributions sont étendues et asymétriques ; les extrêmes visibles doivent être étudiés, mais ne sont pas automatiquement des erreurs.

## 5. Variables catégorielles
Pour chaque variable, le tableau présente le nombre de clients, le nombre de churners et le churn rate. `SeniorCitizen` est traité sémantiquement comme une catégorie binaire.

In [ ]:
categorical_tables = {}
for column in CATEGORICAL_COLUMNS:
    categorical_tables[column] = categorical_summary(data, column)
    print(f'\n### {column}')
    display(categorical_tables[column].style.format({'churn_rate': '{:.2%}'}))

## 6. Relations prioritaires
Le graphique réunit les churn rates des axes définis comme prioritaires. Les écarts ne contrôlent pas les autres variables et ne doivent pas être interprétés causalement.

In [ ]:
plt.figure(figsize=(14, 14))
plt.imshow(plt.imread(FIGURES_DIR / 'key_categorical_churn_rates.png'))
plt.axis('off');

**Associations principales.** Le churn rate est de 42,71 % en contrat mensuel, contre 11,27 % en contrat d'un an et 2,83 % en contrat de deux ans. Il atteint 41,89 % avec la fibre, 41,64 % sans support technique, 41,77 % sans sécurité en ligne, 45,29 % avec paiement par chèque électronique et 33,57 % avec facturation dématérialisée. Ces catégories peuvent se recouvrir fortement.

## 7. Analyse multivariée ciblée
Deux croisements répondent à des questions précises : variation contrat × ancienneté, puis service Internet × contrat. Les charges mensuelles sont aussi comparées par contrat et target.

In [ ]:
contract_tenure, internet_contract = multivariate_tables(data)
display(contract_tenure.style.format({'churn_rate': '{:.2%}'}))
display(internet_contract.style.format({'churn_rate': '{:.2%}'}))
monthly_contract = data.groupby(['Contract', 'Churn'])['MonthlyCharges'].agg(['count', 'mean', 'median'])
display(monthly_contract.round(2))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, ['contract_tenure_churn_heatmap.png', 'internet_contract_churn_heatmap.png', 'monthly_charges_contract_churn.png']):
    ax.imshow(plt.imread(FIGURES_DIR / name)); ax.axis('off')
plt.tight_layout();

**Segments observés.** Le segment contrat mensuel et ancienneté 0–12 mois compte 1 994 clients et affiche 51,35 % de churn. Le segment fibre et contrat mensuel compte 2 128 clients et affiche 54,61 %. À l'inverse, les contrats longs affichent des taux nettement plus faibles dans chaque type d'accès. Ces résultats décrivent des segments associés au churn, sans démontrer que modifier le contrat ou le service changerait causalement le comportement.

## 8. Corrélations numériques
La corrélation de Pearson est calculée uniquement sur les trois variables quantitatives. `SeniorCitizen` est exclu car son stockage numérique représente une catégorie.

In [ ]:
display(data[NUMERIC_COLUMNS].corr().round(3))
plt.figure(figsize=(7, 5))
plt.imshow(plt.imread(FIGURES_DIR / 'numeric_correlations.png'))
plt.axis('off');

`tenure` et `TotalCharges` sont fortement corrélés (0,826), ce qui est attendu pour un cumul dans le temps. `MonthlyCharges` et `TotalCharges` présentent une corrélation de 0,651. Ces relations pourront affecter certains modèles, mais aucune suppression de feature n'est décidée pendant l'EDA.

## 9. Synthèse, limites et implications futures

Les associations les plus marquées concernent le contrat mensuel, la faible ancienneté, la fibre, l'absence de support ou de sécurité en ligne, le chèque électronique et la facturation dématérialisée. D'autres écarts apparaissent pour les seniors, l'absence de partenaire ou de personnes à charge. Le genre et le service téléphonique montrent des écarts faibles dans l'analyse brute.

**Hypothèses de feature engineering, non implémentées :** bandes d'ancienneté, interaction contrat × ancienneté, interaction Internet × contrat, indicateur de services de support/sécurité, nombre de services souscrits, et relation entre charges mensuelles et ancienneté.

**Limites :** dataset fictif, analyse observationnelle, absence de temporalité détaillée et de coûts métier, comparaisons non ajustées pour les facteurs confondants, petits effectifs dans certains croisements, et risque de redondance entre variables de services.